# Cleaning & EDA — full PRCA cohort (File A/B/C)

This notebook loads the **private** Prolific + Qualtrics exports from the sibling data folder (never committed):

```text
../sibling_data/PRCAProlificExport_FileA.csv
../sibling_data/PRCAProlificExport_FileB.csv
../sibling_data/PRCAQualtricsExport_FileC.csv
```

It cleans to an analytic sample, scores ground-truth PRCA subscales, and produces EDA tables that map onto the research questions:

1. **RQ1 — Employment:** does employment status improve CA prediction over demographics alone?
2. **RQ2 — Transit:** does transportation-use data improve prediction / get used sensibly?
3. **RQ3 — Combined:** does employment + transit help beyond either alone, or are cues redundant?
4. **Main RQ — Stereotyping:** does LLM prediction error cluster by demographic group?


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ca_personas.eda import run_eda
from ca_personas.load import load_full_cohort
from ca_personas.paths import (
    DEFAULT_PROLIFIC_A,
    DEFAULT_PROLIFIC_B,
    DEFAULT_QUALTRICS_C,
    sibling_data_available,
)

pd.set_option("display.max_columns", 40)
print("sibling_data available:", sibling_data_available())
print(DEFAULT_PROLIFIC_A)
print(DEFAULT_PROLIFIC_B)
print(DEFAULT_QUALTRICS_C)
assert sibling_data_available(), (
    "Place File A/B/C under ../sibling_data/ before running this notebook."
)


## Load → join → clean → score

- Stack Prolific File A + File B (two recruitment waves, same schema)
- Join to Qualtrics File C on `Q0` ↔ `Participant id`
- Keep inner-join rows with complete PRCA group (`Q1–Q6`) + interpersonal (`Q13–Q18`) items
- Flag employment / transit coverage for the tiered research contrasts


In [ ]:
participants, cleaning_report = load_full_cohort(join_how="inner")
print(json.dumps(cleaning_report, indent=2))
participants.head()


In [ ]:
# Research-covariate coverage (supports RQ1–RQ3 sample support)
coverage = participants[
    ["has_core_demos", "has_employment_info", "has_transit_info", "has_employment_and_transit"]
].mean().rename("share").to_frame()
coverage


## Ground-truth CA distributions

Targets for every persona tier and ML baseline: group + interpersonal PRCA subscales (6–30).


In [ ]:
gt_summary = participants[["gt_group_ca", "gt_interpersonal_ca"]].describe().T
gt_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, col, title in zip(
    axes,
    ["gt_group_ca", "gt_interpersonal_ca"],
    ["Group CA", "Interpersonal CA"],
):
    participants[col].hist(ax=ax, bins=range(6, 32), color="#C5050C", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("PRCA subscale (6–30)")
axes[0].set_ylabel("Participants")
fig.suptitle("Ground-truth CA score distributions")
fig.tight_layout()
plt.show()


## RQ1 lens — CA by employment status

If an LLM stereotypes “unemployed → higher CA,” residual error should later be checked against these empirical means.


In [ ]:
by_emp = (
    participants.groupby("Employment status", dropna=False)
    .agg(
        n=("participant_id", "count"),
        mean_group=("gt_group_ca", "mean"),
        mean_interpersonal=("gt_interpersonal_ca", "mean"),
        pct_high_group=("gt_group_band", lambda s: (s == "high").mean()),
    )
    .reset_index()
)
by_emp


## RQ2 lens — CA by public-transit use (Q26)

Transit items feed the `transit` persona tier. Sparse `Q20`/`Q21` (license/car) is expected in File C; frequency items `Q26–Q29` are the primary signal.


In [ ]:
by_transit = (
    participants.groupby("Q26", dropna=False)
    .agg(
        n=("participant_id", "count"),
        mean_group=("gt_group_ca", "mean"),
        mean_interpersonal=("gt_interpersonal_ca", "mean"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
by_transit


## RQ3 lens — employment × transit contingency

If employment and transit use are strongly associated in this sample, Tier 3 may not beat Tier 1 or 2 alone.


In [ ]:
xtab = pd.crosstab(
    participants["Employment status"].fillna("(missing)"),
    participants["Q26"].fillna("(missing)"),
    margins=True,
)
xtab


## Stereotyping lens — CA by sex and country

Demographic slices used later when correlating LLM absolute error with group membership.


In [ ]:
for col in ("Sex", "Country of residence", "Student status"):
    print("\n===", col, "===")
    display(
        participants.groupby(col, dropna=False)
        .agg(
            n=("participant_id", "count"),
            mean_group=("gt_group_ca", "mean"),
            mean_interpersonal=("gt_interpersonal_ca", "mean"),
        )
        .reset_index()
    )


## Write EDA artifacts

Artifacts land in `outputs/eda/` (gitignored) and `data/processed/` for the prediction pipeline.


In [ ]:
out = ROOT / "outputs" / "eda"
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
participants.to_csv(processed / "participants_scored.csv", index=False)
(processed / "cleaning_report.json").write_text(json.dumps(cleaning_report, indent=2))
artifacts = run_eda(participants, out, cleaning_report=cleaning_report)
artifacts
